Step 1: Import Required Libraries

In [1]:
import pandas as pd

Step 2: Load the Files

In [2]:
ledger_df = pd.read_csv("ledger.csv")
gateway_df = pd.read_csv("gateway_export.csv")

print(ledger_df.shape)
print(gateway_df.shape)

ledger_df.head()
gateway_df.head()

(547, 8)
(530, 8)


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXN100001,287,27,2026-01-04 11:49:00,49,UPI,captured,79
1,TXN100002,122,11,2026-01-24 04:59:00,1499,Wallet,captured,22
2,TXN100003,92,22,2026-01-29 13:01:00,49,Netbanking,captured,100
3,TXN100004,343,16,2026-01-30 13:51:00,99,UPI,captured,34
4,TXN100006,236,23,2026-01-08 06:52:00,99,Wallet,captured,39


Step 3: Create the Reconciliation Function

In [3]:
def reconcile_payments(ledger_df, gateway_df):
    """
    Reconcile ledger records with gateway export.

    Returns:
        missing_in_gateway
        missing_in_ledger
        amount_mismatches
        status_mismatches
    """

    # --------------------------------------------------
    # 1. Identify missing transaction IDs
    # --------------------------------------------------

    ledger_ids = set(ledger_df["transaction_id"])
    gateway_ids = set(gateway_df["transaction_id"])

    missing_in_gateway_ids = ledger_ids - gateway_ids
    missing_in_ledger_ids = gateway_ids - ledger_ids

    missing_in_gateway = ledger_df[ledger_df["transaction_id"].isin(missing_in_gateway_ids)].copy()

    missing_in_ledger = gateway_df[gateway_df["transaction_id"].isin(missing_in_ledger_ids)].copy()

    # --------------------------------------------------
    # 2. Compare common transactions
    # --------------------------------------------------

    merged = pd.merge(ledger_df,gateway_df,on="transaction_id",suffixes=("_ledger", "_gateway"))

    # --------------------------------------------------
    # 3. Amount mismatches
    # --------------------------------------------------

    amount_mismatches = merged[
        merged["amount_inr_ledger"] != merged["amount_inr_gateway"]
    ].copy()

    amount_mismatches["amount_difference"] = (
        amount_mismatches["amount_inr_ledger"]
        - amount_mismatches["amount_inr_gateway"]
    )

    # --------------------------------------------------
    # 4. Status mismatches
    # --------------------------------------------------

    status_mismatches = merged[
        merged["status_ledger"] != merged["status_gateway"]
    ].copy()

    return (
        missing_in_gateway,
        missing_in_ledger,
        amount_mismatches,
        status_mismatches
    )

Step 4: Execute the Function

In [4]:
(
    missing_in_gateway,
    missing_in_ledger,
    amount_mismatches,
    status_mismatches
) = reconcile_payments(
    ledger_df,
    gateway_df
)

Step 5: Report Discrepancy Counts

In [5]:
print("Transactions Missing in Gateway:",
      len(missing_in_gateway))

print("Transactions Missing in Ledger:",
      len(missing_in_ledger))

print("Amount Mismatches:",
      len(amount_mismatches))

print("Status Mismatches:",
      len(status_mismatches))

Transactions Missing in Gateway: 27
Transactions Missing in Ledger: 10
Amount Mismatches: 16
Status Mismatches: 9


Step 6: Display Sample Records

In [6]:
print("\nMissing In Gateway")
display(missing_in_gateway.head())

print("\nMissing In Ledger")
display(missing_in_ledger.head())

print("\nAmount Mismatches")
display(amount_mismatches.head())

print("\nStatus Mismatches")
display(status_mismatches.head())


Missing In Gateway


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXN100000,350,16,2026-01-29 11:13:00,2999,Wallet,captured,85
5,TXN100005,196,3,2026-01-06 22:06:00,99,UPI,captured,60
36,TXN100036,321,16,2026-01-25 16:31:00,49,UPI,captured,35
44,TXN100044,309,15,2026-01-24 23:34:00,99,Netbanking,captured,62
56,TXN100056,132,24,2026-01-07 14:44:00,299,Wallet,captured,21



Missing In Ledger


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
520,TXNX9000,262,25,2026-01-14 00:00:00,149,Netbanking,captured,54
521,TXNX9001,96,32,2026-01-21 00:00:00,4999,Wallet,captured,71
522,TXNX9002,86,32,2026-01-10 00:00:00,149,Wallet,captured,40
523,TXNX9003,231,40,2026-01-02 00:00:00,799,UPI,captured,62
524,TXNX9004,70,13,2026-01-27 00:00:00,1499,Netbanking,captured,52



Amount Mismatches


,transaction_id,user_id_ledger,merchant_id_ledger,transaction_time_ledger,amount_inr_ledger,payment_method_ledger,status_ledger,risk_score_ledger,user_id_gateway,merchant_id_gateway,transaction_time_gateway,amount_inr_gateway,payment_method_gateway,status_gateway,risk_score_gateway,amount_difference
9,TXN100011,306,28,2026-01-09 01:06:00,2999,Card,failed,44,306,28,2026-01-09 01:06:00,2949,Card,failed,44,50
17,TXN100019,292,20,2026-01-24 05:39:00,49,UPI,captured,51,292,20,2026-01-24 05:39:00,99,UPI,captured,51,-50
36,TXN100039,79,14,2026-01-05 07:24:00,1499,UPI,captured,8,79,14,2026-01-05 07:24:00,1399,UPI,captured,8,100
194,TXN100204,59,18,2026-01-12 17:23:00,49,Card,captured,73,59,18,2026-01-12 17:23:00,-51,Card,captured,73,100
207,TXN100218,203,39,2026-01-10 01:14:00,49,UPI,captured,7,203,39,2026-01-10 01:14:00,-1,UPI,captured,7,50



Status Mismatches


,transaction_id,user_id_ledger,merchant_id_ledger,transaction_time_ledger,amount_inr_ledger,payment_method_ledger,status_ledger,risk_score_ledger,user_id_gateway,merchant_id_gateway,transaction_time_gateway,amount_inr_gateway,payment_method_gateway,status_gateway,risk_score_gateway
39,TXN100042,295,25,2026-01-19 22:01:00,1499,UPI,captured,61,295,25,2026-01-19 22:01:00,1499,UPI,failed,61
204,TXN100215,140,14,2026-01-20 14:26:00,299,Wallet,captured,96,140,14,2026-01-20 14:26:00,299,Wallet,failed,96
254,TXN100267,329,31,2026-01-28 08:03:00,299,Wallet,captured,47,329,31,2026-01-28 08:03:00,299,Wallet,failed,47
286,TXN100300,152,20,2026-01-03 10:42:00,49,UPI,captured,57,152,20,2026-01-03 10:42:00,49,UPI,failed,57
371,TXN100392,269,32,2026-01-15 20:34:00,49,Card,captured,17,269,32,2026-01-15 20:34:00,49,Card,failed,17


Step 7: Calculate Percentages for Project Report

In [9]:
total_transactions = len(ledger_df)

print(
    "Missing in Gateway %:",
    round(len(missing_in_gateway) * 100 / total_transactions, 2)
)

print(
    "Missing in Ledger %:",
    round(len(missing_in_ledger) * 100 / total_transactions, 2)
)

print(
    "Amount Mismatch %:",
    round(len(amount_mismatches) * 100 / total_transactions, 2)
)

print(
    "Status Mismatch %:",
    round(len(status_mismatches) * 100 / total_transactions, 2)
)

Missing in Gateway %: 4.94
Missing in Ledger %: 1.83
Amount Mismatch %: 2.93
Status Mismatch %: 1.65
